In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm, tnrange, tqdm_notebook
import time
import sys
import struct
import os
import tqdm

from scipy.optimize import curve_fit

%matplotlib widget
import ipywidgets as widgets
import traitlets
from IPython.display import display
from ipywidgets import widgets
from tkinter import Tk, filedialog
from IPython.display import clear_output
import io
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import AppLayout, Button, Layout

from alive_progress import alive_bar
import time

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = "Arial"

In [2]:
## Exponential decay function

def exp_decay(t,a,tau):
    return a*np.exp(-t/tau)

In [3]:
## Bi-exponential decay function

def bi_exp_decay(t,a1,tau_1,a2,tau_2,t0,c):
    return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c
    #return exp_decay(t,a1,tau_1) + exp_decay(t,a2,tau_2) + c

In [5]:
fol_loc = r"G:\Shared drives\Pauzauskie Team Drive\Users\CG\Lab Work\TB-TCSPC\05172023"
trpl_file_name = r"\default-1_000.ptu"
output_file_name = trpl_file_name

sys.argv[1] = fol_loc + trpl_file_name
sys.argv[2] = fol_loc + output_file_name[:-4] + ".txt"

In [4]:
# Read_PHU.py    Read PicoQuant Unified Histogram Files
# This is demo code. Use at your own risk. No warranties.
# Keno Goertz, PicoQUant GmbH, February 2018

# Note that marker events have a lower time resolution and may therefore appear 
# in the file slightly out of order with respect to regular (photon) event records.
# This is by design. Markers are designed only for relatively coarse 
# synchronization requirements such as image scanning. 

# T Mode data are written to an output file [filename]
# We do not keep it in memory because of the huge amout of memory
# this would take in case of large files. Of course you can change this, 
# e.g. if your files are not too big. 
# Otherwise it is best process the data on the fly and keep only the results.

# Tag Types
tyEmpty8      = struct.unpack(">i", bytes.fromhex("FFFF0008"))[0]
tyBool8       = struct.unpack(">i", bytes.fromhex("00000008"))[0]
tyInt8        = struct.unpack(">i", bytes.fromhex("10000008"))[0]
tyBitSet64    = struct.unpack(">i", bytes.fromhex("11000008"))[0]
tyColor8      = struct.unpack(">i", bytes.fromhex("12000008"))[0]
tyFloat8      = struct.unpack(">i", bytes.fromhex("20000008"))[0]
tyTDateTime   = struct.unpack(">i", bytes.fromhex("21000008"))[0]
tyFloat8Array = struct.unpack(">i", bytes.fromhex("2001FFFF"))[0]
tyAnsiString  = struct.unpack(">i", bytes.fromhex("4001FFFF"))[0]
tyWideString  = struct.unpack(">i", bytes.fromhex("4002FFFF"))[0]
tyBinaryBlob  = struct.unpack(">i", bytes.fromhex("FFFFFFFF"))[0]


sys.argv[1] = r'E:\Shared drives\Pauzauskie Team Drive\CG\Projects\MNVH\Lifetime Data\RGF\5 min scan H3 Bulk Diamond 470 nm Pulsed Laser with 500LP filter 1.phu'
sys.argv[2] = r'E:\Shared drives\Pauzauskie Team Drive\CG\Projects\MNVH\Lifetime Data\5 min scan H3 Bulk Diamond 470 nm Pulsed Laser with 500LP filter 1.txt'

head_i, tail_i = os.path.split(sys.argv[1])
head_o, tail_o = os.path.split(sys.argv[2])

if len(sys.argv) != 3:
    print("USAGE: Read_PHU.py inputfile.PHU outputfile.txt")
    exit(0)

inputfile = open(sys.argv[1], "rb")
outputfile = open(sys.argv[2], "w+")

# Check if inputfile is a valid PHU file
# Python strings don't have terminating NULL characters, so they're stripped
magic = inputfile.read(8).decode("ascii").strip('\0')
if magic != "PQHISTO":
    print("ERROR: Magic invalid, this is not a PHU file.")
    exit(0)

version = inputfile.read(8).decode("ascii").strip('\0')
##outputfile.write("Tag version: %s\n" % version)

# Write the header data to outputfile and also save it in memory.
# There's no do ... while in Python, so an if statement inside the while loop
# breaks out of it
tagDataList = []    # Contains tuples of (tagName, tagValue)
while True:
    tagIdent = inputfile.read(32).decode("ascii").strip('\0')
    tagIdx = struct.unpack("<i", inputfile.read(4))[0]
    tagTyp = struct.unpack("<i", inputfile.read(4))[0]
    if tagIdx > -1:
        evalName = tagIdent + '(' + str(tagIdx) + ')'
    else:
        evalName = tagIdent
    ##outputfile.write("\n%-40s" % evalName)
    if tagTyp == tyEmpty8:
        inputfile.read(8)
        #outputfile.write("<empty Tag>")
        tagDataList.append((evalName, "<empty Tag>"))
    elif tagTyp == tyBool8:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        if tagInt == 0:
            #outputfile.write("False")
            tagDataList.append((evalName, "False"))
        else:
            #outputfile.write("True")
            tagDataList.append((evalName, "True"))
    elif tagTyp == tyInt8:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        #outputfile.write("%d" % tagInt)
        tagDataList.append((evalName, tagInt))
    elif tagTyp == tyBitSet64:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        #outputfile.write("{0:#0{1}x}".format(tagInt,18)) # hex with trailing 0s
        tagDataList.append((evalName, tagInt))
    elif tagTyp == tyColor8:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        #outputfile.write("{0:#0{1}x}".format(tagInt,18)) # hex with trailing 0s
        tagDataList.append((evalName, tagInt))
    elif tagTyp == tyFloat8:
        tagFloat = struct.unpack("<d", inputfile.read(8))[0]
        #outputfile.write("%-3E" % tagFloat)
        tagDataList.append((evalName, tagFloat))
    elif tagTyp == tyFloat8Array:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        #outputfile.write("<Float array with %d entries>" % tagInt/8)
        tagDataList.append((evalName, tagInt))
    elif tagTyp == tyTDateTime:
        tagFloat = struct.unpack("<d", inputfile.read(8))[0]
        tagTime = int((tagFloat - 25569) * 86400)
        tagTime = time.gmtime(tagTime)
        #outputfile.write(time.strftime("%a %b %d %H:%M:%S %Y", tagTime))
        tagDataList.append((evalName, tagTime))
    elif tagTyp == tyAnsiString:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        tagString = inputfile.read(tagInt).decode("ascii").strip("\0")
        #outputfile.write("%s" % tagString)
        tagDataList.append((evalName, tagString))
    elif tagTyp == tyWideString:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        tagString = inputfile.read(tagInt).decode("ascii").strip("\0")
        #outputfile.write("%s" % tagString)
        tagDataList.append((evalName, tagString))
    elif tagTyp == tyBinaryBlob:
        tagInt = struct.unpack("<q", inputfile.read(8))[0]
        #outputfile.write("<Binary blob with %d bytes>" % tagInt)
        tagDataList.append((evalName, tagInt))
    else:
        print("ERROR: Unknown tag type")
        exit(0)
    if tagIdent == "Header_End":
        break

# Reformat the saved data for easier access
tagNames = [tagDataList[i][0] for i in range(0, len(tagDataList))]
tagValues = [tagDataList[i][1] for i in range(0, len(tagDataList))]

# Write histogram data to file
curveIndices = [tagValues[i] for i in range(0, len(tagNames))\
                if tagNames[i][0:-3] == "HistResDscr_CurveIndex"]

histogramBins = tagValues[tagNames.index("HistResDscr_HistogramBins(0)")]

# Empty 2-D array to store intensity values
y_data = np.zeros(shape=(len(curveIndices),histogramBins))


for i in tqdm.notebook.tnrange(len(curveIndices),desc='Total progress'):
    ##outputfile.write("\n-----------------------")
    histogramBins = tagValues[tagNames.index("HistResDscr_HistogramBins(%d)" % i)]
    resolution = tagValues[tagNames.index("HistResDscr_MDescResolution(%d)" % i)]

    y_data_i = []
    x_data = []
    
    for j in tqdm.notebook.tnrange(histogramBins,desc='Curve %d'%(i+1)):
        try:
            histogramData = struct.unpack("<i", inputfile.read(4))[0]
            y_data_i = np.append(y_data_i,histogramData)
            x_data = np.append(x_data,j*resolution*1e9)
            
        except:
            print("The file ended earlier than expected, at bin %d/%d."\
                  % (j, histogramBins))
    
    y_data[i,:] = y_data_i

inputfile.close()
outputfile.close()

USAGE: Read_PHU.py inputfile.PHU outputfile.txt


FileNotFoundError: [Errno 2] No such file or directory: '--ip=127.0.0.1'

: 

In [71]:
plt.figure(figsize=(10,5))
plt.subplot(111)

for i in curveIndices:
    plt.plot(x_data,y_data[i,:],label='Cruve %d'%(i+1))

plt.legend() 
plt.yscale('log')
plt.title('Complete data from TRPL experiment')
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [116]:
def bi_fit_analysis():
    
    time_x = x_data

    start = np.argmin(np.abs(time_x[:]-w_start.value))
    stop = np.argmin(np.abs(time_x[:]-w_stop.value))

    #print(start,stop)

    time_x_edit = time_x[start:stop]

    intes_y = []

    plt.figure(figsize=(10,5))
    plt.subplot(111)


    for i in range(0,1):
        intes_y = y_data[i,start:stop]
        decay_start = np.argmax(intes_y)
        decay_stop = stop

        time_decay = time_x_edit[decay_start:decay_stop]
        decay_curve = intes_y[decay_start:decay_stop]

        norm_decay_curve = (decay_curve-np.min(decay_curve))/(np.max(decay_curve)-np.min(decay_curve))
        #print(norm_decay_curve)

        a1 = w_a1.value
        tau_1 = w_tau_1.value

        a2 = w_a2.value
        tau_2 = w_tau_2.value

        t0 = w_t0.value
        c = w_c.value

        p0 = (a1,tau_1,a2,tau_2,t0,c)

        popt, pcov = curve_fit(bi_exp_decay,time_decay,norm_decay_curve,p0)

        print(popt)

        residuals = norm_decay_curve - bi_exp_decay(time_decay,*popt)
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((norm_decay_curve-np.mean(norm_decay_curve))**2)
        r_squared = 1 - (ss_res / ss_tot)

        #print(r_squared)
        #plt.plot(time_x_edit,norm_intes_y,label='Curve %d'%(i+1))
        #plt.plot(time_x_edit,intes_y,label='Curve %d'%(i+1))

        plt.plot(time_decay,norm_decay_curve,label='Curve %d'%(i+1))
        plt.plot(time_decay,bi_exp_decay(time_decay,*popt),label='Fit %d'%(i+1))

    plt.xlabel('Time(ns)',fontsize=12.0)
    plt.ylabel('Counts',fontsize=12.0)

    plt.yscale('log')
    #plt.title('Data from Multiharp Software' + tail_i,fontsize=15.0)
    #plt.xlim(0,120)

    plt.legend(title='Curves')
    plt.show()

In [127]:
caption = widgets.Label(value='Set the range for x-axis, so that there is only one decay')

w_start = widgets.FloatText(
    value=0,
    description='Start:',
    disabled=False
)

w_stop = widgets.FloatText(
    value=110,
    description='Stop:',
    disabled=False
)


hbox1 = widgets.HBox(items)

items2 = [caption,hbox1]
vbox1 = widgets.VBox(items2)

display(vbox1)

w_method = widgets.RadioButtons(
    options=['mono-exp', 'bi-exp'],
#     value='pineapple',
    description='Fitting method',
    disabled=False
)

display(w_method)

def exp_fit(change):
    
    if w_method.value == 'mono-exp':
        if vbox6 != None:
            vbox6.close()
        show_mono_exp_fit()
    
    elif w_method.value == 'bi-exp':
        if vbox6 != None:
            vbox6.close()
        show_bi_exp_fit()
        
        
def show_mono_exp_fit():
    
    vbox6.close()

    caption2 = widgets.Label(value='Set the initial guess for parameters:')

    w_eqn = widgets.HTMLMath(
        value=r"$$y = a1*exp^{-\frac{(t-t_0)}{\tau_{1}}} + c$$",
        placeholder='Some HTML',
        description='Some HTML',
    )

    w_a1 = widgets.FloatText(
        value=0,
        description='a1:',
        disabled=False
    )

    w_tau_1 = widgets.FloatText(
        value=0,
        description='tau_1:',
        disabled=False
    )

    w_t0 = widgets.FloatText(
        value=0,
        description='t0',
        disabled=False
    )

    w_c = widgets.FloatText(
        value=0,
        description='c:',
        disabled=False
    )

    items3 = [caption2,w_eqn]
    vbox2 = widgets.VBox(items3)

    items4 = [w_a1,w_tau_1]
    items6 = [w_t0,w_c]

    vbox3 = widgets.VBox(items4)
    vbox5 = widgets.VBox(items6)

    hbox2 = widgets.HBox([vbox3,vbox5])

    vbox6 = widgets.VBox([vbox1,vbox2,hbox2])

    display(vbox6)
        
        
def show_bi_exp_fit():

    caption2 = widgets.Label(value='Set the initial guess for parameters:')

    w_eqn = widgets.HTMLMath(
        value=r"$$y = a1*exp^{-\frac{(t-t_0)}{\tau_{1}}} + a2*exp^{-\frac{(t-t_0)}{\tau_{2}}} + c$$",
        placeholder='Some HTML',
        description='Some HTML',
    )

    w_a1 = widgets.FloatText(
        value=0,
        description='a1:',
        disabled=False
    )

    w_tau_1 = widgets.FloatText(
        value=0,
        description='tau_1:',
        disabled=False
    )

    w_a2 = widgets.FloatText(
        value=0,
        description='a1:',
        disabled=False
    )

    w_tau_2 = widgets.FloatText(
        value=0,
        description='tau_2:',
        disabled=False
    )

    w_t0 = widgets.FloatText(
        value=0,
        description='t0',
        disabled=False
    )

    w_c = widgets.FloatText(
        value=0,
        description='c:',
        disabled=False
    )

    items3 = [caption2,w_eqn]
    vbox2 = widgets.VBox(items3)

    items4 = [w_a1,w_tau_1]
    items5 = [w_a2,w_tau_2]
    items6 = [w_t0,w_c]

    vbox3 = widgets.VBox(items4)
    vbox4 = widgets.VBox(items5)
    vbox5 = widgets.VBox(items6)

    hbox2 = widgets.HBox([vbox3,vbox4,vbox5])

    vbox6 = widgets.VBox([vbox1,vbox2,hbox2])

    display(vbox6)

w_method.observe(exp_fit, names='value')

w_plot = widgets.Button(
    description='Run Fitting',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Run fitting analysis',
    #icon='check'
)

def on_button_clicked(b):
    bi_fit_analysis()

w_plot.on_click(on_button_clicked)

display(w_plot)

RadioButtons(description='Fitting method', options=('mono-exp', 'bi-exp'), value='mono-exp')

Button(button_style='success', description='Run Fitting', style=ButtonStyle(), tooltip='Run fitting analysis')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

[0.         0.         0.         0.         0.         0.07653693]


<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c
C:\ProgramData\Anaconda3\lib\site-packages\scipy\optimize\minpack.py:828: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',
<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c
<ipython-input-116-90dcffebe9e6>:14: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  plt.figure(figsize=(10,5))


...

[0.         0.         0.         0.         0.         0.07653693]


<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c
C:\ProgramData\Anaconda3\lib\site-packages\scipy\optimize\minpack.py:828: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',
<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

[0.         0.         0.         0.         0.         0.07653693]


<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c
C:\ProgramData\Anaconda3\lib\site-packages\scipy\optimize\minpack.py:828: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',
<ipython-input-65-b7e857f23224>:4: RuntimeWarning: divide by zero encountered in true_divide
  return a1*np.exp(-(t-t0)/tau_1)+a2*np.exp(-(t-t0)/tau_2) + c


In [4]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 10px 10px 0px',
        padding='5px 5px 5px 5px'
     )

def make_app_layout(h, ls, c, f):

    return AppLayout(header=h,
                     left_sidebar=ls,
                     center=c,
                     right_sidebar=None,
                     footer=f,
                     pane_widths=[2, 3, 0],
                     pane_heights=[0.5, 5, 0.5])

In [57]:
plt.close('all')

class Data_Plotter(widgets.HBox):
    def __init__(self):
        super().__init__()
        output = widgets.Output()

        self.x = x_data
        self.y = y_data[0,:]
        initial_color = '#FF00DD'

        with output:
            self.fig, self.ax = plt.subplots(constrained_layout=True,
                                             figsize=(5, 3.5))
            #plt.autoscale(enable=True, axis='both')
        self.line, = self.ax.plot(self.x, self.y, initial_color)

        self.fig.canvas.toolbar_position = 'bottom'
        self.ax.grid(True)

        ### define plot data widgets

        color_picker = widgets.ColorPicker(value=initial_color,
                                           description='pick a color')

        text_xlabel = widgets.Text(value='',
                                   description='xlabel',
                                   continuous_update=False)

        text_ylabel = widgets.Text(value='',
                                   description='ylabel',
                                   continuous_update=False)

        x_lim = widgets.FloatRangeSlider(
            value=[500, 700],
            min=0,
            max=1000.0,
            step=0.1,
            description='X-axis range:',
            disabled=False,
            continuous_update=True,
            orientation='horizontal',
            readout=True,
            readout_format='.2f',
        )

        y_lim = widgets.FloatRangeSlider(
            value=[0, 500],
            min=0,
            max=1000,
            step=0.1,
            description='Y-axis range:',
            disabled=False,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='.2f',
        )

        plot_data = widgets.Button(
            description='Plot Data',
            disabled=False,
            button_style='',  # 'success', 'info', 'warning', 'danger' or ''
            tooltip='Data plotted',
            icon='check'  # (FontAwesome names without the `fa-` prefix)
        )

        header_button = widgets.Button(description='Welcome to Data Plotter',
                                       button_style='success',
                                       layout=Layout(height='auto',
                                                     width='auto'))
        footer_button = widgets.Button(
            description='Created by Chaman Gupta, PZ Group, UW',
            button_style='success',
            layout=Layout(height='auto', width='auto'))

        #tab_contents = ['P0', 'P1', 'P2', 'P3', 'P4']
        #children_tab = [widgets.Text(description=name) for name in tab_contents]

        
        controls = widgets.VBox([
            color_picker,
            text_xlabel,
            text_ylabel,
            x_lim,
            y_lim,
            plot_data,
        ])

        controls.layout = make_box_layout()

        out_box = widgets.Box([output])
        output.layout = make_box_layout()
        
        ### define data processing widgets
        
        intes_data_check = widgets.Checkbox(value=False,description='Clean lamp data',disabled=False,indent=False)
         
        data_proc = widgets.VBox([intes_data_check])
        data_proc.layout = make_box_layout()
    
        
        tab = widgets.Tab()
        tab.titles = ('Plot Controls','Data Cleaning')
        tab.children = [controls,data_proc]
        

        # observe stuff
        #int_slider.observe(self.update, 'value')
        color_picker.observe(self.line_color, 'value')
        text_xlabel.observe(self.update_xlabel, 'value')
        text_ylabel.observe(self.update_ylabel, 'value')
        x_lim.observe(self.update_xlim, 'value')
        y_lim.observe(self.update_ylim, 'value')
        plot_data.on_click(self.update_selectf)
        intes_data_check.observe(self.f,'value')

        text_xlabel.value = 'x'
        text_ylabel.value = 'y'

        main_app = make_app_layout(header_button, tab, output,footer_button)

        # add to children
        self.children = [main_app]

    def get_data(self, selectHalLamp_1):

        flamp_1 = selectHalLamp_1.files

        #print(flamp_1)

        for i in range(len(flamp_1)):

            head, tail = os.path.split(flamp_1[i])

            data = pd.read_csv(flamp_1[i],
                               sep=None,
                               engine='python',
                               names=['col_a_data', 'col_b_data'])
            x_data = data['col_a_data'].to_numpy()
            y_data = data['col_b_data'].to_numpy()

            self.x = x_data
            self.y = y_data

            #print(self.x)
            #print(self.y)

    def line_color(self, change):
        self.line.set_color(change.new)

    def update_xlabel(self, change):
        self.ax.set_xlabel(change.new)

    def update_ylabel(self, change):
        self.ax.set_ylabel(change.new)

    def update_xlim(self, change):
        self.ax.set_xlim(change.new)

    def update_ylim(self, change):
        self.ax.set_ylim(change.new)

    def update_selectf(self, plot_data):

        self.line, = self.ax.plot(self.x, self.y, color='k')
        plt.autoscale(enable=True, axis='both')
        self.fig.canvas.draw()
  
    def f(self,change):
        if change.new==True:
            style = {'description_width': 'initial'}
            min_val_intensity = widgets.BoundedFloatText(value=550,min=0,max=1000,step=0.1,description='Minimum Intensity',disabled=False,style=style)
            self.data_roc
            #display(min_val_intensity)
            
            
Data_Plotter()

Data_Plotter(children=(AppLayout(children=(Button(button_style='success', description='Welcome to Data Plotter…